# Labs 19 and 20: Comps, calculators and chart reads

*Project LANTERN, Tool Tutorial Part 2. Classroom sample: Project Wren (`deal_a`) and its rival Project Heron (`deal_b`). Fictional, for DAMG 7245.*

In [1]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "params.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
from src.llm import client as llm
from src import schemas, prompts, sample_data as sd
print("repository:", ROOT, "| LLM mode:", llm.MODE)

repository: /Users/sailokeshreddynandavarapu/Documents/Sem_5_Fall_26/Lab_Activity/Lab2/DAMG-7245-Fall-2026 | LLM mode: replay


In [3]:
# From the slide: Calculators that leave a trace
import hashlib, json
def trace(name, value, inputs, formula):
    t = dict(calc=name, value=value, inputs=inputs,
             formula=formula)
    raw = json.dumps(t, sort_keys=True).encode()
    t["calc_id"] = hashlib.sha1(raw).hexdigest()[:12]
    return t

def ltm(fy, cur, prior):             # three facts
    v = fy["value"] + cur["value"] - prior["value"]
    ids = [f["fact_id"] for f in (fy, cur, prior)]
    return trace("ltm", v, ids, "FY + YTD_cur - YTD_prior")

def ev(price, sh, debt, cash, pref=0.0, nci=0.0):
    v = (price * sh["value"] + debt["value"]
         + pref + nci - cash["value"])
    ids = [f["fact_id"] for f in (sh, debt, cash)]
    return trace("ev", v, ids, "P*sh+debt+pref+nci-cash")

In [4]:
import pandas as pd
cf = pd.read_csv("data/reference/comps_facts_sample.csv"); px = pd.read_csv("data/reference/prices_asof_20260930.csv").set_index("ticker")
def fact(t, name, v): return {"fact_id": f"F-{t}-{name}", "value": float(v)}
rows = []
for _, r in cf.iterrows():
    t = r["ticker"]
    rev = ltm(fact(t, "fy_rev", r.fy_revenue), fact(t, "ytd_rev", r.ytd_revenue_current), fact(t, "ytd_rev_prior", r.ytd_revenue_prior))
    ebitda = ltm(fact(t, "fy_ebitda", r.fy_ebitda), fact(t, "ytd_ebitda", r.ytd_ebitda_current), fact(t, "ytd_ebitda_prior", r.ytd_ebitda_prior))
    e = ev(px.loc[t, "close"], fact(t, "shares", r.shares_out), fact(t, "debt", r.debt), fact(t, "cash", r.cash), pref=r.preferred, nci=r.nci)
    evr = trace("ev_revenue", e["value"] / rev["value"], [e["calc_id"], rev["calc_id"]], "EV / LTM revenue")
    eve = trace("ev_ebitda", e["value"] / ebitda["value"], [e["calc_id"], ebitda["calc_id"]], "EV / LTM EBITDA") if ebitda["value"] > 0 else None
    rows.append({"ticker": t, "ltm_revenue": rev["value"], "ltm_revenue_calc_id": rev["calc_id"], "ev": e["value"], "ev_calc_id": e["calc_id"],
                 "ev_revenue": evr["value"], "ev_revenue_calc_id": evr["calc_id"],
                 "ev_ebitda": eve["value"] if eve else None, "ev_ebitda_calc_id": eve["calc_id"] if eve else None})   # n.m. when EBITDA <= 0
comps = pd.DataFrame(rows)
Path("outputs/comps").mkdir(parents=True, exist_ok=True); comps.to_csv("outputs/comps/deal_a.csv", index=False)   # CS2 section 8: a calc_id beside every metric
print(comps[["ticker", "ev_revenue", "ev_revenue_calc_id", "ev_ebitda", "ev_ebitda_calc_id"]].round(2).to_string(index=False))
print(comps[["ev_revenue", "ev_ebitda"]].quantile([.25, .5, .75]).round(2))   # n.m. (negative EBITDA) excluded

ticker  ev_revenue ev_revenue_calc_id  ev_ebitda ev_ebitda_calc_id
  SMPA        3.21       3ce21ffb181b      16.26      131003b20f6e
  SMPB        2.63       68828103b594      19.25      9d430e42e87b
  SMPC        2.78       0af4e43ec171      12.77      4a2bc47981d2
  SMPD        2.93       45458a0d6ab2        NaN               NaN
  SMPE        4.01       b8efc6a7f206      19.80      ed9ac435bec4
  SMPF        3.09       2edb95cad353      19.60      bab9d47c7a52
      ev_revenue  ev_ebitda
0.25        2.81      16.26
0.50        3.01      19.25
0.75        3.18      19.60


In [5]:
# Chart vs table: a recorded chart read against the seller's table
reads = json.load(open("data/dataroom/deal_a/chart_reads.json"))
F = {(f["metric"], f["period"]): f["value"] for f in sd.seller_facts()}
for c in reads:
    gap = (c["value"] - F[(c["metric"], c["period"])]) / F[(c["metric"], c["period"])]
    print(c["chart"], f"chart {c['value']/1e6:.1f}m vs table {F[(c['metric'], c['period'])]/1e6:.1f}m, gap {gap:.1%}",
          "-> DISCREPANCY" if abs(gap) > llm.P["reconcile"]["tolerance_rel"] else "-> ok")

Revenue by fiscal year (CIM, page 4) chart 42.0m vs table 41.2m, gap 1.9% -> DISCREPANCY


**Exercise 19/20.** Trace SMPA's EV/EBITDA back to its input facts with the calc ids. Why is SMPD's EV/EBITDA left out of the quartiles?

In [6]:
# 1. Retrieve SMPA's input data.
r = cf.set_index("ticker").loc["SMPA"]
price = float(px.loc["SMPA", "close"])

# 2. Calculate LTM EBITDA.
ebitda_trace = ltm(
    fact("SMPA", "fy_ebitda", r.fy_ebitda),
    fact("SMPA", "ytd_ebitda", r.ytd_ebitda_current),
    fact("SMPA", "ytd_ebitda_prior", r.ytd_ebitda_prior),
)

# 3. Calculate enterprise value.
ev_trace = ev(
    price,
    fact("SMPA", "shares", r.shares_out),
    fact("SMPA", "debt", r.debt),
    fact("SMPA", "cash", r.cash),
    pref=r.preferred,
    nci=r.nci,
)

# 4. Connect EV/EBITDA to both calculation IDs.
multiple_trace = trace(
    "ev_ebitda",
    ev_trace["value"] / ebitda_trace["value"],
    [ev_trace["calc_id"], ebitda_trace["calc_id"]],
    "EV / LTM EBITDA",
)

for result in [ebitda_trace, ev_trace, multiple_trace]:
    print(json.dumps(result, indent=2))

# Verify this matches the calculation already in the comps table.
saved_id = comps.loc[
    comps["ticker"] == "SMPA", "ev_ebitda_calc_id"
].iloc[0]
assert multiple_trace["calc_id"] == saved_id

# 5. Inspect SMPD's EBITDA.
d = cf.set_index("ticker").loc["SMPD"]
smpd_ebitda = (
    d.fy_ebitda + d.ytd_ebitda_current - d.ytd_ebitda_prior
)

print(f"\nSMPA EV/EBITDA: {multiple_trace['value']:.2f}x")
print(f"SMPD LTM EBITDA: ${smpd_ebitda:,.0f}")

{
  "calc": "ltm",
  "value": 169000000.0,
  "inputs": [
    "F-SMPA-fy_ebitda",
    "F-SMPA-ytd_ebitda",
    "F-SMPA-ytd_ebitda_prior"
  ],
  "formula": "FY + YTD_cur - YTD_prior",
  "calc_id": "d68c65f5ec09"
}
{
  "calc": "ev",
  "value": 2748100000.0,
  "inputs": [
    "F-SMPA-shares",
    "F-SMPA-debt",
    "F-SMPA-cash"
  ],
  "formula": "P*sh+debt+pref+nci-cash",
  "calc_id": "b6630e39a635"
}
{
  "calc": "ev_ebitda",
  "value": 16.26094674556213,
  "inputs": [
    "b6630e39a635",
    "d68c65f5ec09"
  ],
  "formula": "EV / LTM EBITDA",
  "calc_id": "131003b20f6e"
}

SMPA EV/EBITDA: 16.26x
SMPD LTM EBITDA: $-1,000,000


In [ ]:
SMPA’s EV/EBITDA is approximately 16.26×. Its ratio calculation references the EV and LTM EBITDA calculation IDs, which reference the underlying input facts. SMPD has negative LTM EBITDA of −$1 million, so the notebook treats its EV/EBITDA as not meaningful and stores it as missing. Pandas excludes that missing value from the EV/EBITDA quartiles.